<a href="https://colab.research.google.com/github/LucasMartinscode/LucasMartinscode/blob/LucasMartinscode-patch-1/Atividade_aula_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Custom S3 Hook


In [ ]:
import boto3
from airflow.hooks.base import BaseHook
from airflow.models import Variable

class CustomS3Hook(BaseHook):
    def __init__(self, bucket: str, **kwargs) -> None:
        super().__init__()
        self.bucket = bucket
        self.client = boto3.client('s3',
            endpoint_url=Variable.get("AWS_ENDPOINT"),
            aws_access_key_id=Variable.get("AWS_ACCESS_KEY_ID"),
            aws_secret_access_key=Variable.get("AWS_SECRET_ACCESS_KEY"),
            aws_session_token=None,
            config=boto3.session.Config(signature_version='s3v4'),
            verify=False,
            region_name=Variable.get("AWS_REGION", default_var="sa-east-1")
        )

    def put_object(self, key: str, buffer):
        self.client.put_object(Body=buffer, Bucket=self.bucket, Key=f"{key}")

    def get_object(self, key: str):
        response = self.client.get_object(Bucket=self.bucket, Key=key)
        return response.get("Body")


Criando uma função para fazer upload dos arquivos para MINIO

In [ ]:
import os

def upload_files_to_minio(local_path, s3_hook):
    for root, dirs, files in os.walk(local_path):
        for file in files:
            file_path = os.path.join(root, file)
            key = os.path.relpath(file_path, local_path)
            with open(file_path, 'rb') as data:
                s3_hook.put_object(key, data.read())


Tratamento e download


In [ ]:
import requests
from bs4 import BeautifulSoup
import os

BASE_URL = "https://github.com"
REPO_URL = "https://github.com/owid/covid-19-data/tree/master/public/data"
RAW_URL = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data"

def download_file(url, save_path):
    response = requests.get(url)
    with open(save_path, 'wb') as file:
        file.write(response.content)

def get_links_from_github(url):
    page = requests.get(url)
    soup = BeautifulSoup(page.content, 'html.parser')
    links = []
    for link in soup.find_all('a', class_='js-navigation-open Link--primary'):
        href = link.get('href')
        if href and 'tree/master/public/data' in href:
            links.append(BASE_URL + href)
    return links

def download_github_folder(base_url, folder_url, download_path):
    if not os.path.exists(download_path):
        os.makedirs(download_path)
    links = get_links_from_github(folder_url)
    for link in links:
        if '/tree/' in link:
            new_folder_name = link.split('/')[-1]
            new_download_path = os.path.join(download_path, new_folder_name)
            download_github_folder(base_url, link, new_download_path)
        elif '/blob/' in link:
            file_name = link.split('/')[-1]
            file_download_url = link.replace('/blob/', '/raw/')
            save_path = os.path.join(download_path, file_name)
            download_file(file_download_url, save_path)

def download_covid_data():
    download_github_folder(BASE_URL, REPO_URL, '/opt/airflow/downloads/covid_data')


Covid-19 DAGS


In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime
from custom_s3_hook import CustomS3Hook
from download_covid_data import download_covid_data, upload_files_to_minio

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'start_date': datetime(2021, 1, 1),
    'retries': 1,
}

def execute_upload():
    local_path = '/opt/airflow/downloads/covid_data'
    s3_hook = CustomS3Hook(bucket="covid-data")
    upload_files_to_minio(local_path, s3_hook)

with DAG(
    dag_id='covid_data_to_minio_dag',
    default_args=default_args,
    description='A DAG to download COVID-19 data and upload to MinIO',
    schedule_interval='@daily',
    catchup=False,
) as dag:

    download_task = PythonOperator(
        task_id='download_covid_data',
        python_callable=download_covid_data,
    )

    upload_task = PythonOperator(
        task_id='upload_to_minio',
        python_callable=execute_upload,
    )

    download_task >> upload_task
